In [0]:
import urllib.request
import os
import shutil
from datetime import date, datetime, timezone
from dateutil.relativedelta import relativedelta

# Obtains the year-month for 2 months prior to the current month in yyyy-MM format
two_months_ago = date.today() - relativedelta(months=2)
formatted_date = two_months_ago.strftime("%Y-%m")

# Define the local directory for this date's data
dir_path = f"/Volumes/nyctaxi/00_landing/data_sources/nyctaxi_yellow/{formatted_date}"

# Define the full path for the downloaded file
local_path = f"{dir_path}/yellow_tripdata_{formatted_date}.parquet"

# Flag to track if we have a valid file to process downstream
file_ready = False

try:
    # Check if the file already exists
    dbutils.fs.ls(local_path)
    print("File already exists in storage volume. Skipping download, proceeding to process tables.")
    file_ready = True
except:
    try:
        # Construct the URL for the Parquet file corresponding to this month
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{formatted_date}.parquet"
        
        # Open a connection and stream the remote file
        response = urllib.request.urlopen(url)
        
        # Create the local directory for this date's data
        os.makedirs(dir_path, exist_ok=True)
        
        # Save the streamed content to the local file in binary mode
        with open(local_path, 'wb') as f:
            shutil.copyfileobj(response, f) # Copy data from response to file
            
        print("File successfully downloaded fresh in current run.")
        file_ready = True
    except Exception as e:
        print(f"File download failed: {str(e)}")

# --- CRITICAL PIPELINE FIX ---
# If the file is ready (whether newly downloaded or pre-existing), ALWAYS tell downstream tasks to run!
if file_ready:
    dbutils.jobs.taskValues.set(key="continue_downstream", value="yes")
else:
    dbutils.jobs.taskValues.set(key="continue_downstream", value="no")